# Add Historical Polymarket Trading-Activity Features to the Final Model-Ready Dataset

## Purpose

This notebook extends the **existing final dissertation model-ready dataset** with historically reconstructable Polymarket market-microstructure information.

It does **not** change:

- the three selected Polymarket competitions;
- the 428 competitive Event IDs;
- the financial target;
- the winner definition;
- the existing probability features;
- the no-look-ahead design.

The new data come primarily from Polymarket's public historical **Trades API**.

---

# What can be reconstructed historically?

## Added to the dataset

### 1. Trading volume / activity

- previous-day share volume;
- previous-day derived notional volume (`size × price`);
- log previous-day volume;
- 7-day volume;
- volume surge;
- relative volume share among competing alternatives.

### 2. Trade count

- previous-day number of trades;
- log trade count;
- relative trade-count share.

### 3. Breadth of participation

Because the public trade extraction is requested with `takerOnly=true`, this is specifically:

- number of unique **takers**, rather than claiming to observe every unique market participant;
- log unique takers;
- relative taker share.

### 4. Directional trading pressure

Using the taker's BUY/SELL side:

`OrderFlowImbalance = (Buy Notional - Sell Notional) / Total Notional`

Range:

- +1 = entirely taker BUY activity;
- 0 = balanced;
- -1 = entirely taker SELL activity.

### 5. Trade size

- average trade notional;
- median trade notional.

### 6. Trader concentration

From each wallet's share of daily mapped notional:

- HHI concentration;
- top-taker share;
- effective number of takers.

These measure whether activity is broadly distributed or concentrated among a small number of active takers.

### 7. Intraday trade-price dispersion proxy

- daily trade-price standard deviation;
- daily trade-price range.

This is **not labelled as true belief disagreement**. It is only a historical price-dispersion proxy because changing transaction prices may reflect both disagreement and information arriving during the day.

### 8. Event-level probability uncertainty

Using the already-existing lagged Polymarket probabilities:

- normalized probability entropy;
- maximum probability / confidence;
- top-two probability gap.

These require no additional API call.

---

# What is deliberately NOT used as a historical predictor?

The notebook does **not** insert current:

- Gamma liquidity;
- bid-ask spread;
- order-book depth;
- open interest;
- holder concentration

into historical rows.

Those quantities can be queried now, but the public documented endpoints do not provide the historical daily snapshots needed for our 2025–2026 modelling period.

Using a current snapshot as though it were known on an old prediction date would be methodologically invalid.

---

# Important modelling principle

This notebook creates a **richer candidate-feature dataset**.

It does **not** mean that every new variable should be placed into the model together.

The recommended later experiment is controlled:

1. Baseline
2. Baseline + Probability
3. Baseline + Volume
4. Baseline + Relative Volume Share
5. Baseline + Unique Takers
6. Baseline + Order-Flow Imbalance
7. Baseline + Trade-Price Dispersion
8. Probability + the strongest activity feature(s)

This avoids data mining and lets us identify which type of Polymarket information, if any, adds value.


## Step 1 — Upload the Existing Final Model-Ready Dataset

Upload:

`Final_Competitive_Markets_MODEL_READY_Dataset.xlsx`


In [ ]:
# ============================================================
# STEP 1: Upload Existing Model-Ready Dataset
# ============================================================

from google.colab import files

uploaded = files.upload()

print("Uploaded files:")
for file_name in uploaded.keys():
    print(" -", file_name)


## Step 2 — Install and Import Libraries


In [ ]:
# ============================================================
# STEP 2: Install and Import Libraries
# ============================================================

!pip -q install requests pandas numpy openpyxl

import ast
import json
import time
from collections import deque
from pathlib import Path

import numpy as np
import pandas as pd
import requests

from IPython.display import display

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 240)

print("Libraries imported successfully.")


## Step 3 — Load and Validate the Existing Model-Ready Dataset


In [ ]:
# ============================================================
# STEP 3: Load Existing Model-Ready Dataset
# ============================================================

MODEL_PATH = (
    "/content/"
    "Final_Competitive_Markets_MODEL_READY_Dataset.xlsx"
)

master = pd.read_excel(
    MODEL_PATH,
    sheet_name="Model_Ready_Master"
)

for col in [
    "Date",
    "Next_Financial_Date",
    "Financial_Prev1_Date",
    "Financial_Prev2_Date",
    "Financial_Prev3_Date",
    "PM_Signal_Date_Lag1"
]:
    if col in master.columns:
        master[col] = pd.to_datetime(
            master[col],
            errors="coerce"
        )

master = (
    master
    .sort_values(
        [
            "Market_ID",
            "Date",
            "Event_ID",
            "Asset"
        ]
    )
    .reset_index(drop=True)
)

print("Rows:", len(master))
print("Competitive events:", master["Event_ID"].nunique())

print("\nEvents by market:")
print(
    master.groupby(
        "Market_ID"
    )["Event_ID"].nunique()
)

assert (
    master.groupby(
        "Event_ID"
    )["Winner"].sum() == 1
).all()

assert master.duplicated(
    [
        "Event_ID",
        "Asset"
    ]
).sum() == 0

print("\nPASS: original event structure is valid.")

display(master.head())


## Step 4 — Define the Same Three Polymarket Events

These are exactly the event slugs used in the earlier full-history probability extraction.


In [ ]:
# ============================================================
# STEP 4: Same Dissertation Polymarket Events
# ============================================================

GAMMA_API = "https://gamma-api.polymarket.com"
DATA_API_TRADES = "https://data-api.polymarket.com/trades"

EVENT_CONFIG = [
    {
        "event_slug":
            "bitcoin-vs-gold-vs-sp-500-in-2026",
        "project_market_id":
            "BTC_GOLD_SP500",
        "group_id":
            "Best_Asset_2026",
        "type":
            "THREE_WAY"
    },
    {
        # Displayed Polymarket title:
        # "Will Bitcoin outperform Gold in 2025?"
        # Actual slug used in the original extraction:
        "event_slug":
            "will-bitcoin-outperform-gold-in-july",
        "project_market_id":
            "BTC_GOLD_2025",
        "group_id":
            "BTC_Outperform_Gold_2025",
        "type":
            "BINARY"
    },
    {
        "event_slug":
            "will-bitcoin-outperform-gold-in-2026",
        "project_market_id":
            "BTC_GOLD_2026",
        "group_id":
            "BTC_Outperform_Gold_2026",
        "type":
            "BINARY"
    }
]

REQUEST_PAUSE = 0.15

for cfg in EVENT_CONFIG:
    print(
        cfg["project_market_id"],
        "->",
        cfg["event_slug"]
    )


## Step 5 — API Helper Functions


In [ ]:
# ============================================================
# STEP 5: API Helpers
# ============================================================

def safe_json_list(value):

    if value is None:
        return []

    if isinstance(value, list):
        return value

    if isinstance(value, tuple):
        return list(value)

    if not isinstance(value, str):
        return []

    value = value.strip()

    if value == "":
        return []

    try:
        parsed = json.loads(value)

        return (
            parsed
            if isinstance(parsed, list)
            else []
        )

    except (
        json.JSONDecodeError,
        TypeError
    ):
        pass

    try:
        parsed = ast.literal_eval(value)

        return (
            parsed
            if isinstance(parsed, list)
            else []
        )

    except (
        ValueError,
        SyntaxError
    ):
        return []


def parse_datetime_utc(value):

    return pd.to_datetime(
        value,
        utc=True,
        errors="coerce"
    )


def get_json(
    url,
    params=None,
    max_retries=6,
    timeout=90
):

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json"
    }

    last_error = None

    for attempt in range(
        1,
        max_retries + 1
    ):

        try:

            response = requests.get(
                url,
                params=params,
                headers=headers,
                timeout=timeout
            )

            if response.status_code == 429:

                wait = min(
                    attempt * 10,
                    60
                )

                print(
                    f"Rate limit — waiting {wait}s"
                )

                time.sleep(wait)
                continue

            if response.status_code in {
                500,
                502,
                503,
                504
            }:

                wait = min(
                    attempt * 5,
                    30
                )

                print(
                    "Temporary server error",
                    response.status_code,
                    f"— waiting {wait}s"
                )

                time.sleep(wait)
                continue

            response.raise_for_status()

            return response.json()

        except requests.RequestException as error:

            last_error = error

            if attempt < max_retries:

                time.sleep(
                    min(
                        attempt * 5,
                        30
                    )
                )

    raise RuntimeError(
        "API request failed after retries: "
        + str(last_error)
    )


## Step 6 — Resolve Child Markets and YES/NO Token IDs

For each selected Polymarket market, we retrieve:

- condition ID;
- CLOB token IDs;
- YES/NO outcome mapping;
- candidate asset mapping.

### Alternative mapping

#### Binary BTC vs Gold

- YES token → Bitcoin
- NO token → Gold

#### BTC vs Gold vs S&P 500

Each candidate has its own child market.

For direct alternative-level activity:

- candidate YES token → that candidate.

The NO token of each three-way child market is retained in the raw mapping for audit, but is not treated as another competing asset.


In [ ]:
# ============================================================
# STEP 6: Resolve Markets + Tokens
# ============================================================

market_rows = []
token_rows = []

for cfg in EVENT_CONFIG:

    event = get_json(
        f"{GAMMA_API}/events/slug/"
        f"{cfg['event_slug']}"
    )

    markets = event.get(
        "markets",
        []
    )

    if not markets:
        raise ValueError(
            "No markets returned for "
            + cfg["event_slug"]
        )

    for market in markets:

        outcomes = safe_json_list(
            market.get("outcomes")
        )

        token_ids = safe_json_list(
            market.get("clobTokenIds")
        )

        if (
            len(outcomes) == 0
            or len(outcomes)
            != len(token_ids)
        ):
            continue

        question = str(
            market.get(
                "question",
                ""
            )
        ).strip()

        group_title = str(
            market.get(
                "groupItemTitle",
                ""
            )
        ).strip()

        include_market = False
        candidate_asset = None

        # ---------------------------------------
        # Three-way event
        # ---------------------------------------

        if cfg["type"] == "THREE_WAY":

            candidate_map = {
                "Bitcoin":
                    "Bitcoin",
                "Gold":
                    "Gold",
                "S&P 500":
                    "SP500"
            }

            if group_title in candidate_map:

                include_market = True

                candidate_asset = (
                    candidate_map[
                        group_title
                    ]
                )

        # ---------------------------------------
        # Binary 2025
        # ---------------------------------------

        elif (
            cfg["project_market_id"]
            == "BTC_GOLD_2025"
        ):

            if (
                "bitcoin outperform gold"
                in question.casefold()
                and "2025" in question
            ):
                include_market = True

        # ---------------------------------------
        # Binary 2026
        # ---------------------------------------

        elif (
            cfg["project_market_id"]
            == "BTC_GOLD_2026"
        ):

            if (
                "bitcoin outperform gold"
                in question.casefold()
                and "2026" in question
            ):
                include_market = True

        if not include_market:
            continue

        condition_id = str(
            market.get(
                "conditionId"
            )
        )

        gamma_market_id = str(
            market.get(
                "id"
            )
        )

        market_rows.append({
            "Project_Market_ID":
                cfg[
                    "project_market_id"
                ],
            "Market_Type":
                cfg["type"],
            "Candidate_Asset":
                candidate_asset,
            "Event_ID_Gamma":
                str(
                    event.get(
                        "id"
                    )
                ),
            "Event_Slug":
                event.get(
                    "slug"
                ),
            "Event_Title":
                event.get(
                    "title"
                ),
            "Gamma_Market_ID":
                gamma_market_id,
            "Condition_ID":
                condition_id,
            "Question":
                question,
            "Group_Item_Title":
                group_title,
            "Market_Start_UTC":
                market.get(
                    "startDate"
                ),
            "Market_Created_UTC":
                (
                    market.get(
                        "createdAt"
                    )
                    or market.get(
                        "creationDate"
                    )
                ),
            "Accepting_Orders_UTC":
                market.get(
                    "acceptingOrdersTimestamp"
                ),
            "Market_End_UTC":
                market.get(
                    "endDate"
                ),
            # Current snapshots: audit only
            "Gamma_Current_Volume":
                market.get(
                    "volume"
                ),
            "Gamma_Current_Volume24hr":
                market.get(
                    "volume24hr"
                ),
            "Gamma_Current_Liquidity":
                market.get(
                    "liquidity"
                )
        })

        for outcome, token_id in zip(
            outcomes,
            token_ids
        ):

            outcome_clean = str(
                outcome
            ).strip()

            alternative_asset = None

            # Binary:
            # YES = Bitcoin
            # NO  = Gold
            if cfg["type"] == "BINARY":

                if (
                    outcome_clean.casefold()
                    == "yes"
                ):

                    alternative_asset = (
                        "Bitcoin"
                    )

                elif (
                    outcome_clean.casefold()
                    == "no"
                ):

                    alternative_asset = (
                        "Gold"
                    )

            # Three-way:
            # direct competing-alternative token
            # is the candidate's YES token.
            elif cfg["type"] == "THREE_WAY":

                if (
                    outcome_clean.casefold()
                    == "yes"
                ):

                    alternative_asset = (
                        candidate_asset
                    )

            token_rows.append({
                "Project_Market_ID":
                    cfg[
                        "project_market_id"
                    ],
                "Market_Type":
                    cfg["type"],
                "Candidate_Asset":
                    candidate_asset,
                "Condition_ID":
                    condition_id,
                "Gamma_Market_ID":
                    gamma_market_id,
                "Token_ID":
                    str(
                        token_id
                    ),
                "Outcome":
                    outcome_clean,
                "Alternative_Asset":
                    alternative_asset
            })


market_info = pd.DataFrame(
    market_rows
)

token_map = pd.DataFrame(
    token_rows
)

# ----------------------
# Validation
# ----------------------

for m in [
    "BTC_GOLD_2025",
    "BTC_GOLD_2026"
]:

    n = (
        market_info[
            "Project_Market_ID"
        ].eq(m).sum()
    )

    assert n == 1, (
        f"Expected one binary market for {m}; "
        f"found {n}."
    )


three_assets = set(
    market_info.loc[
        market_info[
            "Project_Market_ID"
        ] == "BTC_GOLD_SP500",
        "Candidate_Asset"
    ]
    .dropna()
)

assert three_assets == {
    "Bitcoin",
    "Gold",
    "SP500"
}

print("Resolved markets:")
display(
    market_info[
        [
            "Project_Market_ID",
            "Market_Type",
            "Candidate_Asset",
            "Question",
            "Condition_ID"
        ]
    ]
)

print("\nToken mapping:")
display(
    token_map[
        [
            "Project_Market_ID",
            "Candidate_Asset",
            "Outcome",
            "Token_ID",
            "Alternative_Asset"
        ]
    ]
)


## Step 7 — Determine the Historical Trade Window

We request enough calendar history to construct lagged and rolling activity features before the first model event.

The final extraction end is the final model date plus one day.


In [ ]:
# ============================================================
# STEP 7: Historical Trade Boundaries
# ============================================================

project_ranges = (
    master.groupby(
        "Market_ID"
    )["Date"]
    .agg(
        Model_First_Date="min",
        Model_Last_Date="max"
    )
    .reset_index()
    .rename(
        columns={
            "Market_ID":
                "Project_Market_ID"
        }
    )
)

market_info = market_info.merge(
    project_ranges,
    on="Project_Market_ID",
    how="left"
)


def choose_start(row):

    model_start = pd.Timestamp(
        row[
            "Model_First_Date"
        ],
        tz="UTC"
    ) - pd.Timedelta(
        days=15
    )

    market_candidates = []

    for col in [
        "Accepting_Orders_UTC",
        "Market_Start_UTC",
        "Market_Created_UTC"
    ]:

        dt = parse_datetime_utc(
            row[col]
        )

        if not pd.isna(dt):
            market_candidates.append(
                dt
            )

    if market_candidates:

        actual_market_start = min(
            market_candidates
        )

        return max(
            actual_market_start,
            model_start
        )

    return model_start


def choose_end(row):

    return (
        pd.Timestamp(
            row[
                "Model_Last_Date"
            ],
            tz="UTC"
        )
        + pd.Timedelta(
            days=1
        )
    )


market_info[
    "Requested_Start_UTC"
] = market_info.apply(
    choose_start,
    axis=1
)

market_info[
    "Requested_End_UTC"
] = market_info.apply(
    choose_end,
    axis=1
)

display(
    market_info[
        [
            "Project_Market_ID",
            "Candidate_Asset",
            "Condition_ID",
            "Requested_Start_UTC",
            "Requested_End_UTC"
        ]
    ]
)


## Step 8 — Robust Historical Trade Downloader

The Data API allows market-scoped trade requests with:

- condition ID;
- start timestamp;
- end timestamp;
- up to 10,000 rows in a window.

The downloader:

1. begins with 7-day windows;
2. requests `takerOnly=true`;
3. if a window returns the 10,000-row cap, automatically splits that window;
4. records failed windows;
5. refuses to silently interpret failed downloads as zero volume.


In [ ]:
# ============================================================
# STEP 8A: One Trade-Window Request
# ============================================================

def request_trade_window(
    condition_id,
    start_utc,
    end_utc,
    limit=10000
):

    start_ts = int(
        start_utc.timestamp()
    )

    # treat requested window as [start, end)
    end_ts = (
        int(
            end_utc.timestamp()
        )
        - 1
    )

    if end_ts < start_ts:
        return []

    params = {
        "market":
            str(
                condition_id
            ),
        "start":
            start_ts,
        "end":
            end_ts,
        "limit":
            limit,
        "offset":
            0,
        "takerOnly":
            "true"
    }

    payload = get_json(
        DATA_API_TRADES,
        params=params
    )

    if not isinstance(
        payload,
        list
    ):

        raise RuntimeError(
            "Unexpected trades payload: "
            + str(
                type(payload)
            )
        )

    return payload


In [ ]:
# ============================================================
# STEP 8B: Auto-Splitting Downloader
# ============================================================

def make_windows(
    start_utc,
    end_utc,
    days=7
):

    windows = []

    current = start_utc

    while current < end_utc:

        nxt = min(
            current
            + pd.Timedelta(
                days=days
            ),
            end_utc
        )

        windows.append(
            (
                current,
                nxt
            )
        )

        current = nxt

    return windows


def fetch_condition_trades(
    condition_id,
    start_utc,
    end_utc,
    initial_days=7,
    minimum_minutes=30
):

    queue = deque(
        make_windows(
            start_utc,
            end_utc,
            days=initial_days
        )
    )

    frames = []
    failures = []

    while queue:

        w_start, w_end = (
            queue.popleft()
        )

        try:

            payload = (
                request_trade_window(
                    condition_id,
                    w_start,
                    w_end
                )
            )

        except Exception as error:

            failures.append({
                "Condition_ID":
                    condition_id,
                "Window_Start":
                    w_start,
                "Window_End":
                    w_end,
                "Error":
                    str(
                        error
                    )
            })

            continue

        # Could be truncated:
        # split this window.
        if len(payload) >= 10000:

            minutes = (
                w_end
                - w_start
            ).total_seconds() / 60

            if minutes <= minimum_minutes:

                failures.append({
                    "Condition_ID":
                        condition_id,
                    "Window_Start":
                        w_start,
                    "Window_End":
                        w_end,
                    "Error":
                        (
                            "Still hit 10,000-row cap "
                            "at minimum window."
                        )
                })

                continue

            midpoint = (
                w_start
                + (
                    w_end
                    - w_start
                ) / 2
            )

            queue.appendleft(
                (
                    midpoint,
                    w_end
                )
            )

            queue.appendleft(
                (
                    w_start,
                    midpoint
                )
            )

            print(
                "10,000-row cap -> splitting:",
                w_start,
                "to",
                w_end
            )

            continue

        if payload:

            temp = pd.DataFrame(
                payload
            )

            temp[
                "Requested_Condition_ID"
            ] = str(
                condition_id
            )

            frames.append(
                temp
            )

        time.sleep(
            REQUEST_PAUSE
        )

    if frames:

        trades = pd.concat(
            frames,
            ignore_index=True
        )

    else:

        trades = pd.DataFrame()

    failures = pd.DataFrame(
        failures
    )

    return trades, failures


## Step 9 — Download the Historical Trade Records


In [ ]:
# ============================================================
# STEP 9: Download Historical Trades
# ============================================================

condition_ranges = (
    market_info[
        [
            "Condition_ID",
            "Requested_Start_UTC",
            "Requested_End_UTC"
        ]
    ]
    .drop_duplicates(
        subset=[
            "Condition_ID"
        ]
    )
    .reset_index(
        drop=True
    )
)

all_trade_frames = []
all_failure_frames = []

for _, row in condition_ranges.iterrows():

    condition_id = row[
        "Condition_ID"
    ]

    print(
        "\nDownloading:",
        condition_id
    )

    print(
        row[
            "Requested_Start_UTC"
        ],
        "to",
        row[
            "Requested_End_UTC"
        ]
    )

    trades_i, failures_i = (
        fetch_condition_trades(
            condition_id,
            row[
                "Requested_Start_UTC"
            ],
            row[
                "Requested_End_UTC"
            ]
        )
    )

    print(
        "Rows:",
        len(
            trades_i
        )
    )

    if not trades_i.empty:
        all_trade_frames.append(
            trades_i
        )

    if not failures_i.empty:
        all_failure_frames.append(
            failures_i
        )


raw_trades = (
    pd.concat(
        all_trade_frames,
        ignore_index=True
    )
    if all_trade_frames
    else pd.DataFrame()
)

failed_windows = (
    pd.concat(
        all_failure_frames,
        ignore_index=True
    )
    if all_failure_frames
    else pd.DataFrame(
        columns=[
            "Condition_ID",
            "Window_Start",
            "Window_End",
            "Error"
        ]
    )
)

print(
    "\nDownloaded trade rows:",
    len(
        raw_trades
    )
)

print(
    "Failed windows:",
    len(
        failed_windows
    )
)

if not failed_windows.empty:

    display(
        failed_windows
    )

    raise RuntimeError(
        "Trade extraction has failed windows. "
        "Rerun Step 9 before continuing. "
        "Do not convert missing API history "
        "into zero volume."
    )

print("PASS: no failed trade windows.")


## Step 10 — Clean and Deduplicate the Trades

We derive trade notional as:

`Trade Notional = size × price`

The resulting value is therefore explicitly called **derived notional**, rather than claiming it is a native historical Gamma volume field.


In [ ]:
# ============================================================
# STEP 10: Clean Trade Data
# ============================================================

expected_cols = {
    "proxyWallet",
    "side",
    "asset",
    "conditionId",
    "size",
    "price",
    "timestamp",
    "outcome",
    "transactionHash"
}

missing_cols = (
    expected_cols
    - set(
        raw_trades.columns
    )
)

if missing_cols:

    raise ValueError(
        "Missing expected trade fields: "
        + str(
            missing_cols
        )
    )


trades = raw_trades.copy()

trades[
    "Condition_ID"
] = trades[
    "conditionId"
].astype(
    str
)

trades[
    "Token_ID"
] = trades[
    "asset"
].astype(
    str
)

trades[
    "Trade_Size_Shares"
] = pd.to_numeric(
    trades[
        "size"
    ],
    errors="coerce"
)

trades[
    "Trade_Price"
] = pd.to_numeric(
    trades[
        "price"
    ],
    errors="coerce"
)

trades[
    "Timestamp"
] = pd.to_numeric(
    trades[
        "timestamp"
    ],
    errors="coerce"
)

trades[
    "Datetime_UTC"
] = pd.to_datetime(
    trades[
        "Timestamp"
    ],
    unit="s",
    utc=True,
    errors="coerce"
)

trades[
    "Date"
] = (
    trades[
        "Datetime_UTC"
    ]
    .dt
    .tz_convert(
        "UTC"
    )
    .dt
    .tz_localize(
        None
    )
    .dt
    .normalize()
)

trades[
    "Side"
] = (
    trades[
        "side"
    ]
    .astype(
        str
    )
    .str
    .upper()
)

trades[
    "Derived_Notional"
] = (
    trades[
        "Trade_Size_Shares"
    ]
    *
    trades[
        "Trade_Price"
    ]
)

trades = trades.dropna(
    subset=[
        "Condition_ID",
        "Token_ID",
        "Datetime_UTC",
        "Trade_Size_Shares",
        "Trade_Price"
    ]
).copy()

trades = trades[
    trades[
        "Trade_Size_Shares"
    ].ge(
        0
    )
    &
    trades[
        "Trade_Price"
    ].between(
        0,
        1,
        inclusive="both"
    )
    &
    trades[
        "Side"
    ].isin(
        [
            "BUY",
            "SELL"
        ]
    )
].copy()

dedup_cols = [
    "transactionHash",
    "Condition_ID",
    "Token_ID",
    "Timestamp",
    "Trade_Size_Shares",
    "Trade_Price",
    "Side",
    "proxyWallet"
]

before = len(
    trades
)

trades = (
    trades
    .drop_duplicates(
        subset=dedup_cols,
        keep="last"
    )
    .sort_values(
        "Datetime_UTC"
    )
    .reset_index(
        drop=True
    )
)

print(
    "Rows before deduplication:",
    before
)

print(
    "Rows after deduplication:",
    len(
        trades
    )
)

print(
    "Duplicates removed:",
    before
    - len(
        trades
    )
)

display(
    trades[
        [
            "Date",
            "Condition_ID",
            "Token_ID",
            "Side",
            "Trade_Size_Shares",
            "Trade_Price",
            "Derived_Notional",
            "proxyWallet"
        ]
    ].head()
)


## Step 11 — Map Trades to the Competitive Alternatives

### Binary markets

- YES-token activity → Bitcoin
- NO-token activity → Gold

### Three-way market

- Bitcoin child-market YES-token activity → Bitcoin
- Gold child-market YES-token activity → Gold
- S&P 500 child-market YES-token activity → SP500

This keeps the variables alternative-specific, which is essential for Conditional Logit.

A market-wide volume value copied to every alternative in an Event ID would be constant within the choice set and therefore could not directly explain which alternative wins.


In [ ]:
# ============================================================
# STEP 11: Map Trades to Alternatives
# ============================================================

mapped_trades = trades.merge(
    token_map[
        [
            "Project_Market_ID",
            "Condition_ID",
            "Token_ID",
            "Outcome",
            "Alternative_Asset"
        ]
    ],
    on=[
        "Condition_ID",
        "Token_ID"
    ],
    how="left",
    validate="many_to_one"
)

model_trades = mapped_trades[
    mapped_trades[
        "Alternative_Asset"
    ].notna()
].copy()

print(
    "Total cleaned trades:",
    len(
        mapped_trades
    )
)

print(
    "Alternative-mapped trades:",
    len(
        model_trades
    )
)

print(
    "\nMapped records:"
)

display(
    model_trades.groupby(
        [
            "Project_Market_ID",
            "Alternative_Asset"
        ]
    )
    .size()
    .reset_index(
        name="Trade_Records"
    )
)


## Step 12 — Build Daily Wallet-Level Activity for Concentration Measures

For every market-alternative-date, we first calculate each active taker's total derived notional.

From this we later calculate:

- top taker share;
- HHI concentration;
- effective number of takers.


In [ ]:
# ============================================================
# STEP 12: Wallet-Level Daily Activity
# ============================================================

wallet_daily = (
    model_trades
    .dropna(
        subset=[
            "proxyWallet"
        ]
    )
    .groupby(
        [
            "Project_Market_ID",
            "Alternative_Asset",
            "Date",
            "proxyWallet"
        ],
        as_index=False
    )
    .agg(
        Wallet_Notional=(
            "Derived_Notional",
            "sum"
        )
    )
)


def concentration_metrics(g):

    total = g[
        "Wallet_Notional"
    ].sum()

    if (
        total <= 0
        or len(
            g
        ) == 0
    ):

        return pd.Series({
            "PM_Taker_HHI_Daily_Audit":
                np.nan,
            "PM_Top_Taker_Share_Daily_Audit":
                np.nan,
            "PM_Effective_Takers_Daily_Audit":
                np.nan
        })

    shares = (
        g[
            "Wallet_Notional"
        ]
        / total
    )

    hhi = float(
        np.sum(
            shares ** 2
        )
    )

    return pd.Series({
        "PM_Taker_HHI_Daily_Audit":
            hhi,
        "PM_Top_Taker_Share_Daily_Audit":
            float(
                shares.max()
            ),
        "PM_Effective_Takers_Daily_Audit":
            (
                1 / hhi
                if hhi > 0
                else np.nan
            )
    })


concentration_daily = (
    wallet_daily
    .groupby(
        [
            "Project_Market_ID",
            "Alternative_Asset",
            "Date"
        ]
    )
    .apply(
        concentration_metrics
    )
    .reset_index()
)

display(
    concentration_daily.head()
)


## Step 13 — Aggregate the Main Historical Trade Features to Daily Frequency


In [ ]:
# ============================================================
# STEP 13: Daily Trading-Activity Features
# ============================================================

# Directional notional components
model_trades[
    "Buy_Notional"
] = np.where(
    model_trades[
        "Side"
    ] == "BUY",
    model_trades[
        "Derived_Notional"
    ],
    0.0
)

model_trades[
    "Sell_Notional"
] = np.where(
    model_trades[
        "Side"
    ] == "SELL",
    model_trades[
        "Derived_Notional"
    ],
    0.0
)


def daily_trade_aggregation(g):

    total_notional = (
        g[
            "Derived_Notional"
        ].sum()
    )

    buy_notional = (
        g[
            "Buy_Notional"
        ].sum()
    )

    sell_notional = (
        g[
            "Sell_Notional"
        ].sum()
    )

    if total_notional > 0:

        imbalance = (
            buy_notional
            - sell_notional
        ) / total_notional

        vwap = np.average(
            g[
                "Trade_Price"
            ],
            weights=g[
                "Derived_Notional"
            ]
        )

    else:

        imbalance = 0.0
        vwap = np.nan

    prices = g[
        "Trade_Price"
    ].dropna()

    if len(prices) >= 2:

        price_std = float(
            prices.std(
                ddof=1
            )
        )

    else:

        price_std = np.nan

    if len(prices) >= 1:

        price_range = float(
            prices.max()
            - prices.min()
        )

    else:

        price_range = np.nan

    return pd.Series({
        "PM_Share_Volume_Daily_Audit":
            g[
                "Trade_Size_Shares"
            ].sum(),

        "PM_Notional_Volume_Daily_Audit":
            total_notional,

        "PM_Trade_Count_Daily_Audit":
            len(
                g
            ),

        "PM_Unique_Takers_Daily_Audit":
            g[
                "proxyWallet"
            ].nunique(
                dropna=True
            ),

        "PM_Buy_Notional_Daily_Audit":
            buy_notional,

        "PM_Sell_Notional_Daily_Audit":
            sell_notional,

        "PM_OrderFlow_Imbalance_Daily_Audit":
            imbalance,

        "PM_Mean_Trade_Notional_Daily_Audit":
            g[
                "Derived_Notional"
            ].mean(),

        "PM_Median_Trade_Notional_Daily_Audit":
            g[
                "Derived_Notional"
            ].median(),

        "PM_VWAP_Daily_Audit":
            vwap,

        "PM_TradePrice_Std_Daily_Audit":
            price_std,

        "PM_TradePrice_Range_Daily_Audit":
            price_range
    })


daily_activity_observed = (
    model_trades
    .groupby(
        [
            "Project_Market_ID",
            "Alternative_Asset",
            "Date"
        ]
    )
    .apply(
        daily_trade_aggregation
    )
    .reset_index()
)

daily_activity_observed = (
    daily_activity_observed.merge(
        concentration_daily,
        on=[
            "Project_Market_ID",
            "Alternative_Asset",
            "Date"
        ],
        how="left",
        validate="one_to_one"
    )
)

print(
    "Observed daily activity rows:",
    len(
        daily_activity_observed
    )
)

display(
    daily_activity_observed.head(
        15
    )
)


## Step 14 — Create a Complete Calendar

For count/volume variables, a successfully downloaded calendar day with no mapped trades is a genuine **zero-activity day**.

Therefore the following are set to zero on no-trade days:

- volume;
- trade count;
- unique takers;
- BUY/SELL notional;
- order-flow imbalance.

Price dispersion and trader-concentration measures are left missing on days where they cannot be observed.

This distinction prevents us from pretending that “no trades” means “zero disagreement”.


In [ ]:
# ============================================================
# STEP 14: Complete Calendar + Genuine Zero Activity
# ============================================================

asset_sets = {
    "BTC_GOLD_2025":
        [
            "Bitcoin",
            "Gold"
        ],
    "BTC_GOLD_2026":
        [
            "Bitcoin",
            "Gold"
        ],
    "BTC_GOLD_SP500":
        [
            "Bitcoin",
            "Gold",
            "SP500"
        ]
}

calendar_frames = []

for market_id, assets in asset_sets.items():

    model_dates = master.loc[
        master[
            "Market_ID"
        ] == market_id,
        "Date"
    ]

    dates = pd.date_range(
        start=(
            model_dates.min()
            - pd.Timedelta(
                days=15
            )
        ),
        end=model_dates.max(),
        freq="D"
    )

    for asset in assets:

        calendar_frames.append(
            pd.DataFrame({
                "Project_Market_ID":
                    market_id,
                "Alternative_Asset":
                    asset,
                "Date":
                    dates
            })
        )


daily_activity = (
    pd.concat(
        calendar_frames,
        ignore_index=True
    )
    .merge(
        daily_activity_observed,
        on=[
            "Project_Market_ID",
            "Alternative_Asset",
            "Date"
        ],
        how="left"
    )
)

zero_activity_cols = [
    "PM_Share_Volume_Daily_Audit",
    "PM_Notional_Volume_Daily_Audit",
    "PM_Trade_Count_Daily_Audit",
    "PM_Unique_Takers_Daily_Audit",
    "PM_Buy_Notional_Daily_Audit",
    "PM_Sell_Notional_Daily_Audit",
    "PM_OrderFlow_Imbalance_Daily_Audit"
]

daily_activity[
    zero_activity_cols
] = daily_activity[
    zero_activity_cols
].fillna(
    0
)

daily_activity = (
    daily_activity
    .sort_values(
        [
            "Project_Market_ID",
            "Alternative_Asset",
            "Date"
        ]
    )
    .reset_index(
        drop=True
    )
)

print(
    "Complete calendar rows:",
    len(
        daily_activity
    )
)

display(
    daily_activity.head(
        15
    )
)


## Step 15 — Create Leakage-Safe Lagged Trading-Activity Features

All predictors are based on calendar information available strictly before the financial prediction date.

### Primary activity candidates

- `PM_Log_Notional_Volume_Lag1`
- `PM_Relative_Volume_Share_Lag1`
- `PM_Log_Unique_Takers_Lag1`
- `PM_OrderFlow_Imbalance_Lag1`

### Additional robustness candidates

- `PM_Log_Notional_Volume_7D_Lag1`
- `PM_Volume_Surge_Lag1`
- `PM_Log_Trade_Count_Lag1`
- `PM_Mean_Trade_Notional_Lag1`
- `PM_Taker_HHI_Lag1`
- `PM_Top_Taker_Share_Lag1`
- `PM_TradePrice_Std_Lag1`
- `PM_TradePrice_Range_Lag1`

The same-day versions remain audit only.


In [ ]:
# ============================================================
# STEP 15A: Lagged Activity Features
# ============================================================

group_cols = [
    "Project_Market_ID",
    "Alternative_Asset"
]

daily_activity[
    "PM_Activity_Signal_Date_Lag1"
] = (
    daily_activity
    .groupby(
        group_cols
    )["Date"]
    .shift(
        1
    )
)


lag_source_map = {
    "PM_Share_Volume_Daily_Audit":
        "PM_Share_Volume_Lag1",

    "PM_Notional_Volume_Daily_Audit":
        "PM_Notional_Volume_Lag1",

    "PM_Trade_Count_Daily_Audit":
        "PM_Trade_Count_Lag1",

    "PM_Unique_Takers_Daily_Audit":
        "PM_Unique_Takers_Lag1",

    "PM_OrderFlow_Imbalance_Daily_Audit":
        "PM_OrderFlow_Imbalance_Lag1",

    "PM_Mean_Trade_Notional_Daily_Audit":
        "PM_Mean_Trade_Notional_Lag1",

    "PM_Median_Trade_Notional_Daily_Audit":
        "PM_Median_Trade_Notional_Lag1",

    "PM_VWAP_Daily_Audit":
        "PM_VWAP_Lag1_Audit",

    "PM_TradePrice_Std_Daily_Audit":
        "PM_TradePrice_Std_Lag1",

    "PM_TradePrice_Range_Daily_Audit":
        "PM_TradePrice_Range_Lag1",

    "PM_Taker_HHI_Daily_Audit":
        "PM_Taker_HHI_Lag1",

    "PM_Top_Taker_Share_Daily_Audit":
        "PM_Top_Taker_Share_Lag1",

    "PM_Effective_Takers_Daily_Audit":
        "PM_Effective_Takers_Lag1"
}


for source_col, lag_col in lag_source_map.items():

    daily_activity[
        lag_col
    ] = (
        daily_activity
        .groupby(
            group_cols
        )[source_col]
        .shift(
            1
        )
    )


# Log transformations
daily_activity[
    "PM_Log_Notional_Volume_Lag1"
] = np.log1p(
    daily_activity[
        "PM_Notional_Volume_Lag1"
    ]
)

daily_activity[
    "PM_Log_Trade_Count_Lag1"
] = np.log1p(
    daily_activity[
        "PM_Trade_Count_Lag1"
    ]
)

daily_activity[
    "PM_Log_Unique_Takers_Lag1"
] = np.log1p(
    daily_activity[
        "PM_Unique_Takers_Lag1"
    ]
)

daily_activity[
    "PM_Log_Mean_Trade_Notional_Lag1"
] = np.log1p(
    daily_activity[
        "PM_Mean_Trade_Notional_Lag1"
    ]
)


In [ ]:
# ============================================================
# STEP 15B: 7-Day Volume + Volume Surge
# ============================================================

# Work separately for every alternative
rolling_frames = []

for (
    market_id,
    asset
), g in daily_activity.groupby(
    group_cols
):

    g = g.sort_values(
        "Date"
    ).copy()

    # Seven calendar days ending yesterday
    g[
        "PM_Notional_Volume_7D_Lag1"
    ] = (
        g[
            "PM_Notional_Volume_Daily_Audit"
        ]
        .shift(
            1
        )
        .rolling(
            window=7,
            min_periods=1
        )
        .sum()
    )

    g[
        "PM_Log_Notional_Volume_7D_Lag1"
    ] = np.log1p(
        g[
            "PM_Notional_Volume_7D_Lag1"
        ]
    )

    # Yesterday's log volume
    lag1_log = np.log1p(
        g[
            "PM_Notional_Volume_Daily_Audit"
        ].shift(
            1
        )
    )

    # Average log volume over days t-8 ... t-2
    previous_7_before_yesterday = (
        np.log1p(
            g[
                "PM_Notional_Volume_Daily_Audit"
            ]
        )
        .shift(
            2
        )
        .rolling(
            window=7,
            min_periods=1
        )
        .mean()
    )

    g[
        "PM_Volume_Surge_Lag1"
    ] = (
        lag1_log
        - previous_7_before_yesterday
    )

    rolling_frames.append(
        g
    )


daily_activity = (
    pd.concat(
        rolling_frames,
        ignore_index=True
    )
    .sort_values(
        [
            "Project_Market_ID",
            "Alternative_Asset",
            "Date"
        ]
    )
    .reset_index(
        drop=True
    )
)


## Step 16 — Create Relative Activity Shares Across the Competing Alternatives

Absolute volume can differ greatly across contracts.

For every market-date, we therefore also calculate the alternative's share of total activity:

`Alternative previous-day volume / total previous-day volume across alternatives`

This is particularly useful for the competitive-choice framing.

If there was no activity for any alternative on the previous day, the activity share is set to the neutral uniform value:

- 1/2 for binary markets;
- 1/3 for the three-way market.


In [ ]:
# ============================================================
# STEP 16: Relative Activity Features
# ============================================================

def add_relative_share(
    frame,
    value_col,
    output_col
):

    out = frame.copy()

    totals = (
        out.groupby(
            [
                "Project_Market_ID",
                "Date"
            ]
        )[value_col]
        .transform(
            "sum"
        )
    )

    choice_size = (
        out.groupby(
            [
                "Project_Market_ID",
                "Date"
            ]
        )[
            "Alternative_Asset"
        ]
        .transform(
            "nunique"
        )
    )

    out[
        output_col
    ] = np.where(
        totals > 0,
        out[
            value_col
        ] / totals,
        1 / choice_size
    )

    return out


daily_activity = add_relative_share(
    daily_activity,
    "PM_Notional_Volume_Lag1",
    "PM_Relative_Volume_Share_Lag1"
)

daily_activity = add_relative_share(
    daily_activity,
    "PM_Trade_Count_Lag1",
    "PM_Relative_TradeCount_Share_Lag1"
)

daily_activity = add_relative_share(
    daily_activity,
    "PM_Unique_Takers_Lag1",
    "PM_Relative_Taker_Share_Lag1"
)

display(
    daily_activity[
        [
            "Project_Market_ID",
            "Alternative_Asset",
            "Date",
            "PM_Log_Notional_Volume_Lag1",
            "PM_Relative_Volume_Share_Lag1",
            "PM_Log_Unique_Takers_Lag1",
            "PM_OrderFlow_Imbalance_Lag1"
        ]
    ].head(
        20
    )
)


## Step 17 — Validate No-Look-Ahead Timing


In [ ]:
# ============================================================
# STEP 17: Timing Validation
# ============================================================

valid = daily_activity[
    "PM_Activity_Signal_Date_Lag1"
].notna()

assert (
    daily_activity.loc[
        valid,
        "PM_Activity_Signal_Date_Lag1"
    ]
    <
    daily_activity.loc[
        valid,
        "Date"
    ]
).all()

print(
    "PASS: every trading-activity signal "
    "comes from a date strictly before "
    "the corresponding event date."
)


## Step 18 — Merge the Historical Activity Features into the Existing Dataset


In [ ]:
# ============================================================
# STEP 18: Merge to Existing Model-Ready Dataset
# ============================================================

activity_for_merge = (
    daily_activity.rename(
        columns={
            "Project_Market_ID":
                "Market_ID",
            "Alternative_Asset":
                "Asset"
        }
    )
)

activity_cols = [
    "Market_ID",
    "Asset",
    "Date",

    # same-day audit
    "PM_Share_Volume_Daily_Audit",
    "PM_Notional_Volume_Daily_Audit",
    "PM_Trade_Count_Daily_Audit",
    "PM_Unique_Takers_Daily_Audit",
    "PM_OrderFlow_Imbalance_Daily_Audit",
    "PM_Mean_Trade_Notional_Daily_Audit",
    "PM_Median_Trade_Notional_Daily_Audit",
    "PM_VWAP_Daily_Audit",
    "PM_TradePrice_Std_Daily_Audit",
    "PM_TradePrice_Range_Daily_Audit",
    "PM_Taker_HHI_Daily_Audit",
    "PM_Top_Taker_Share_Daily_Audit",
    "PM_Effective_Takers_Daily_Audit",

    # timing audit
    "PM_Activity_Signal_Date_Lag1",

    # leakage-safe activity
    "PM_Share_Volume_Lag1",
    "PM_Notional_Volume_Lag1",
    "PM_Log_Notional_Volume_Lag1",
    "PM_Notional_Volume_7D_Lag1",
    "PM_Log_Notional_Volume_7D_Lag1",
    "PM_Volume_Surge_Lag1",

    "PM_Trade_Count_Lag1",
    "PM_Log_Trade_Count_Lag1",

    "PM_Unique_Takers_Lag1",
    "PM_Log_Unique_Takers_Lag1",

    "PM_OrderFlow_Imbalance_Lag1",

    "PM_Mean_Trade_Notional_Lag1",
    "PM_Median_Trade_Notional_Lag1",
    "PM_Log_Mean_Trade_Notional_Lag1",

    "PM_VWAP_Lag1_Audit",

    "PM_TradePrice_Std_Lag1",
    "PM_TradePrice_Range_Lag1",

    "PM_Taker_HHI_Lag1",
    "PM_Top_Taker_Share_Lag1",
    "PM_Effective_Takers_Lag1",

    # relative activity
    "PM_Relative_Volume_Share_Lag1",
    "PM_Relative_TradeCount_Share_Lag1",
    "PM_Relative_Taker_Share_Lag1"
]

activity_for_merge = (
    activity_for_merge[
        activity_cols
    ]
)

master_plus = master.merge(
    activity_for_merge,
    on=[
        "Market_ID",
        "Asset",
        "Date"
    ],
    how="left",
    validate="one_to_one"
)

assert len(
    master_plus
) == len(
    master
)

assert master_plus[
    "Event_ID"
].nunique() == master[
    "Event_ID"
].nunique()

assert (
    master_plus.groupby(
        "Event_ID"
    )["Winner"].sum() == 1
).all()

print(
    "Original rows:",
    len(
        master
    )
)

print(
    "Rows after activity merge:",
    len(
        master_plus
    )
)

print(
    "Events after merge:",
    master_plus[
        "Event_ID"
    ].nunique()
)

print(
    "\nPASS: the original modelling sample "
    "has not been altered."
)


## Step 19 — Add Event-Level Probability Uncertainty Features

These are derived from the already-existing `PM_Prob_Lag1`.

### Normalized entropy

- 0 = one alternative dominates completely;
- 1 = probabilities are maximally even.

This is an **uncertainty/concentration measure**, not proof of trader disagreement.

### Maximum probability

The probability assigned to the event's current favourite.

### Probability gap

Difference between the largest and second-largest probabilities.

These are event-context variables. They are constant across alternatives inside one Event ID, so:

- they may be usable in non-choice ML;
- they cannot identify a standalone alternative effect in ordinary Conditional Logit because they do not vary within the event.


In [ ]:
# ============================================================
# STEP 19: Event-Level Probability Uncertainty
# ============================================================

def event_probability_context(g):

    p = (
        g[
            "PM_Prob_Lag1"
        ]
        .astype(
            float
        )
        .to_numpy()
    )

    p = np.clip(
        p,
        1e-15,
        None
    )

    p = p / p.sum()

    entropy = -np.sum(
        p * np.log(
            p
        )
    )

    normalized_entropy = (
        entropy
        / np.log(
            len(
                p
            )
        )
    )

    sorted_p = np.sort(
        p
    )[::-1]

    if len(
        sorted_p
    ) >= 2:

        gap = (
            sorted_p[0]
            - sorted_p[1]
        )

    else:

        gap = np.nan

    return pd.Series({
        "PM_Event_Entropy_Lag1":
            normalized_entropy,
        "PM_Event_MaxProb_Lag1":
            sorted_p[0],
        "PM_Event_Top2_Gap_Lag1":
            gap
    })


event_context = (
    master_plus
    .groupby(
        "Event_ID"
    )
    .apply(
        event_probability_context
    )
    .reset_index()
)

master_plus = master_plus.merge(
    event_context,
    on="Event_ID",
    how="left",
    validate="many_to_one"
)

display(
    master_plus[
        [
            "Market_ID",
            "Event_ID",
            "Date",
            "Asset",
            "PM_Prob_Lag1",
            "PM_Event_Entropy_Lag1",
            "PM_Event_MaxProb_Lag1",
            "PM_Event_Top2_Gap_Lag1"
        ]
    ].head(
        12
    )
)


## Step 20 — Coverage and Variation Checks

Before modelling, we need to know:

1. Are the activity features available for the final 428-event sample?
2. How many no-trade days exist?
3. Do the alternative-specific features actually vary within Event IDs?
4. How complete are the optional price-dispersion and concentration measures?


In [ ]:
# ============================================================
# STEP 20A: Coverage Summary
# ============================================================

coverage = (
    master_plus.groupby(
        [
            "Market_ID",
            "Asset"
        ]
    )
    .agg(
        Rows=(
            "Event_ID",
            "size"
        ),

        First_Date=(
            "Date",
            "min"
        ),

        Last_Date=(
            "Date",
            "max"
        ),

        Volume_Lag_Available=(
            "PM_Log_Notional_Volume_Lag1",
            lambda x:
                x.notna().sum()
        ),

        Zero_Volume_Days=(
            "PM_Notional_Volume_Lag1",
            lambda x:
                x.eq(
                    0
                ).sum()
        ),

        Taker_Lag_Available=(
            "PM_Log_Unique_Takers_Lag1",
            lambda x:
                x.notna().sum()
        ),

        Price_Dispersion_Available=(
            "PM_TradePrice_Range_Lag1",
            lambda x:
                x.notna().sum()
        ),

        HHI_Available=(
            "PM_Taker_HHI_Lag1",
            lambda x:
                x.notna().sum()
        )
    )
    .reset_index()
)

coverage[
    "Volume_Coverage_Rate"
] = (
    coverage[
        "Volume_Lag_Available"
    ]
    /
    coverage[
        "Rows"
    ]
)

coverage[
    "Price_Dispersion_Coverage_Rate"
] = (
    coverage[
        "Price_Dispersion_Available"
    ]
    /
    coverage[
        "Rows"
    ]
)

coverage[
    "HHI_Coverage_Rate"
] = (
    coverage[
        "HHI_Available"
    ]
    /
    coverage[
        "Rows"
    ]
)

display(
    coverage
)


In [ ]:
# ============================================================
# STEP 20B: Within-Event Variation
# ============================================================

variation_features = [
    "PM_Log_Notional_Volume_Lag1",
    "PM_Relative_Volume_Share_Lag1",
    "PM_Log_Unique_Takers_Lag1",
    "PM_OrderFlow_Imbalance_Lag1",
    "PM_TradePrice_Range_Lag1"
]

variation_rows = []

for feature in variation_features:

    n_unique = (
        master_plus.groupby(
            "Event_ID"
        )[feature]
        .nunique(
            dropna=True
        )
    )

    variation_rows.append({
        "Feature":
            feature,

        "Events_Total":
            len(
                n_unique
            ),

        "Events_With_Within_Event_Variation":
            (
                n_unique > 1
            ).sum(),

        "Events_Without_Within_Event_Variation":
            (
                n_unique <= 1
            ).sum()
    })


variation_summary = pd.DataFrame(
    variation_rows
)

display(
    variation_summary
)


## Step 21 — Feature Guide and Recommended Use

This table is deliberately conservative.

It separates:

- primary features worth testing first;
- robustness features;
- audit-only fields;
- event-context features that are not standalone Conditional Logit predictors.


In [ ]:
# ============================================================
# STEP 21: Feature Guide
# ============================================================

feature_guide = pd.DataFrame([
    {
        "Feature":
            "PM_Log_Notional_Volume_Lag1",
        "Information_Type":
            "Trading activity / volume",
        "Priority":
            "PRIMARY",
        "Conditional_Logit":
            "Yes, if it varies within Event_ID",
        "Meaning":
            (
                "Log of previous-calendar-day "
                "alternative-specific derived notional."
            )
    },
    {
        "Feature":
            "PM_Relative_Volume_Share_Lag1",
        "Information_Type":
            "Relative attention",
        "Priority":
            "PRIMARY",
        "Conditional_Logit":
            "Yes",
        "Meaning":
            (
                "Alternative's share of previous-day "
                "mapped trading activity."
            )
    },
    {
        "Feature":
            "PM_Log_Unique_Takers_Lag1",
        "Information_Type":
            "Participation breadth",
        "Priority":
            "PRIMARY",
        "Conditional_Logit":
            "Yes",
        "Meaning":
            (
                "Log number of distinct active taker "
                "wallets on previous calendar day."
            )
    },
    {
        "Feature":
            "PM_OrderFlow_Imbalance_Lag1",
        "Information_Type":
            "Directional trading pressure",
        "Priority":
            "PRIMARY",
        "Conditional_Logit":
            "Yes",
        "Meaning":
            (
                "(taker BUY notional - taker SELL notional) "
                "/ total notional; bounded [-1,1]."
            )
    },
    {
        "Feature":
            "PM_TradePrice_Range_Lag1",
        "Information_Type":
            "Price dispersion / volatility proxy",
        "Priority":
            "SECONDARY",
        "Conditional_Logit":
            "Yes when observed and varying",
        "Meaning":
            (
                "Previous-day max minus min transaction price. "
                "Do not call this true trader disagreement."
            )
    },
    {
        "Feature":
            "PM_TradePrice_Std_Lag1",
        "Information_Type":
            "Price dispersion / volatility proxy",
        "Priority":
            "SECONDARY",
        "Conditional_Logit":
            "Yes when observed and varying",
        "Meaning":
            (
                "Previous-day standard deviation "
                "of transaction prices."
            )
    },
    {
        "Feature":
            "PM_Taker_HHI_Lag1",
        "Information_Type":
            "Trader concentration",
        "Priority":
            "SECONDARY",
        "Conditional_Logit":
            "Yes when observed and varying",
        "Meaning":
            (
                "HHI of mapped daily notional "
                "across active taker wallets."
            )
    },
    {
        "Feature":
            "PM_Top_Taker_Share_Lag1",
        "Information_Type":
            "Trader concentration",
        "Priority":
            "SECONDARY",
        "Conditional_Logit":
            "Yes when observed and varying",
        "Meaning":
            (
                "Largest taker's share of mapped "
                "daily notional."
            )
    },
    {
        "Feature":
            "PM_Log_Mean_Trade_Notional_Lag1",
        "Information_Type":
            "Trade size",
        "Priority":
            "SECONDARY",
        "Conditional_Logit":
            "Yes when observed",
        "Meaning":
            "Log average trade notional on t-1."
    },
    {
        "Feature":
            "PM_Log_Notional_Volume_7D_Lag1",
        "Information_Type":
            "Persistent activity",
        "Priority":
            "ROBUSTNESS",
        "Conditional_Logit":
            "Yes",
        "Meaning":
            "Log seven-day notional ending at t-1."
    },
    {
        "Feature":
            "PM_Volume_Surge_Lag1",
        "Information_Type":
            "Information-arrival / activity shock",
        "Priority":
            "ROBUSTNESS",
        "Conditional_Logit":
            "Yes",
        "Meaning":
            (
                "Yesterday's log volume relative "
                "to the preceding seven-day norm."
            )
    },
    {
        "Feature":
            "PM_Event_Entropy_Lag1",
        "Information_Type":
            "Event uncertainty",
        "Priority":
            "SECONDARY",
        "Conditional_Logit":
            "NO as standalone predictor",
        "Meaning":
            (
                "Normalized entropy of lagged "
                "competitive probabilities."
            )
    },
    {
        "Feature":
            "PM_Event_MaxProb_Lag1",
        "Information_Type":
            "Event confidence",
        "Priority":
            "SECONDARY",
        "Conditional_Logit":
            "NO as standalone predictor",
        "Meaning":
            (
                "Highest lagged probability "
                "within the competitive event."
            )
    },
    {
        "Feature":
            "PM_Event_Top2_Gap_Lag1",
        "Information_Type":
            "Event confidence",
        "Priority":
            "SECONDARY",
        "Conditional_Logit":
            "NO as standalone predictor",
        "Meaning":
            (
                "Gap between first- and second-ranked "
                "lagged Polymarket probabilities."
            )
    }
])

display(
    feature_guide
)


## Step 22 — Explicitly Record Features We Cannot Retrospectively Use

This protects the dissertation from accidental look-ahead or false historical data.


In [ ]:
# ============================================================
# STEP 22: Unavailable Historical Snapshot Features
# ============================================================

not_historical = pd.DataFrame([
    {
        "Candidate_Feature":
            "Liquidity / order-book depth",
        "API_Status":
            "Current public snapshot available",
        "Historical_Model_Use":
            "NO",
        "Reason":
            (
                "No historical daily order-book snapshot "
                "series is used here."
            )
    },
    {
        "Candidate_Feature":
            "Bid-ask spread / best bid / best ask",
        "API_Status":
            "Current public order-book endpoints available",
        "Historical_Model_Use":
            "NO",
        "Reason":
            (
                "Current spread cannot be assigned "
                "to old prediction dates."
            )
    },
    {
        "Candidate_Feature":
            "Open interest",
        "API_Status":
            "Current Data API endpoint available",
        "Historical_Model_Use":
            "NO",
        "Reason":
            (
                "No timestamped historical OI series "
                "is used here."
            )
    },
    {
        "Candidate_Feature":
            "Holder concentration",
        "API_Status":
            "Current holders endpoint available",
        "Historical_Model_Use":
            "NO",
        "Reason":
            (
                "Current holders are not historical "
                "holder snapshots."
            )
    },
    {
        "Candidate_Feature":
            "True trader belief disagreement",
        "API_Status":
            "Not directly observed",
        "Historical_Model_Use":
            "NO direct measure",
        "Reason":
            (
                "Trade-price dispersion and two-sided "
                "flow can only be treated as proxies."
            )
    }
])

display(
    not_historical
)


## Step 23 — Final QC Summary


In [ ]:
# ============================================================
# STEP 23: QC Summary
# ============================================================

qc_summary = pd.DataFrame([
    {
        "Check":
            "Original rows",
        "Value":
            len(
                master
            )
    },
    {
        "Check":
            "Original Event IDs",
        "Value":
            master[
                "Event_ID"
            ].nunique()
    },
    {
        "Check":
            "Rows after merge",
        "Value":
            len(
                master_plus
            )
    },
    {
        "Check":
            "Event IDs after merge",
        "Value":
            master_plus[
                "Event_ID"
            ].nunique()
    },
    {
        "Check":
            "Downloaded cleaned trades",
        "Value":
            len(
                trades
            )
    },
    {
        "Check":
            "Mapped alternative-level trades",
        "Value":
            len(
                model_trades
            )
    },
    {
        "Check":
            "Failed API windows",
        "Value":
            len(
                failed_windows
            )
    },
    {
        "Check":
            "Rows missing primary volume lag",
        "Value":
            master_plus[
                "PM_Log_Notional_Volume_Lag1"
            ].isna().sum()
    },
    {
        "Check":
            "Rows missing price-dispersion range",
        "Value":
            master_plus[
                "PM_TradePrice_Range_Lag1"
            ].isna().sum()
    },
    {
        "Check":
            "Rows missing taker HHI",
        "Value":
            master_plus[
                "PM_Taker_HHI_Lag1"
            ].isna().sum()
    }
])

display(
    qc_summary
)


## Step 24 — Export the New Model-Ready Dataset

Output workbook:

`Final_Competitive_Markets_MODEL_READY_with_PM_Trading_Activity.xlsx`

Sheets:

- `Model_Ready_Master`
- `Daily_PM_Activity`
- `Coverage`
- `Within_Event_Variation`
- `Market_Info`
- `Token_Map`
- `Feature_Guide`
- `Unavailable_Historical`
- `QC_Summary`
- `Failed_API_Windows`

The full raw trade records are saved separately as a compressed CSV.


In [ ]:
# ============================================================
# STEP 24: Export — FIXED FOR EXCEL TIMEZONES
# ============================================================

OUTPUT_XLSX = (
    "/content/"
    "Final_Competitive_Markets_MODEL_READY_"
    "with_PM_Trading_Activity.xlsx"
)

RAW_TRADES_OUTPUT = (
    "/content/"
    "Polymarket_Selected_Markets_"
    "Historical_Trades.csv.gz"
)


# ------------------------------------------------------------
# Helper: make any dataframe safe for Excel
# Excel does not support timezone-aware datetimes
# ------------------------------------------------------------

def make_excel_safe(df):

    df = df.copy()

    for col in df.columns:

        # Proper pandas timezone-aware datetime column
        if isinstance(
            df[col].dtype,
            pd.DatetimeTZDtype
        ):
            df[col] = (
                df[col]
                .dt
                .tz_localize(None)
            )

        # Object columns may also contain timezone-aware
        # Timestamp objects
        elif df[col].dtype == "object":

            has_tz_timestamp = df[col].apply(
                lambda x:
                    isinstance(x, pd.Timestamp)
                    and x.tzinfo is not None
            ).any()

            if has_tz_timestamp:

                df[col] = df[col].apply(
                    lambda x:
                        x.tz_localize(None)
                        if (
                            isinstance(
                                x,
                                pd.Timestamp
                            )
                            and x.tzinfo
                            is not None
                        )
                        else x
                )

    return df


# ------------------------------------------------------------
# Prepare Excel-safe versions
# ------------------------------------------------------------

master_plus_export = make_excel_safe(
    master_plus
)

daily_export = (
    daily_activity.rename(
        columns={
            "Project_Market_ID":
                "Market_ID",
            "Alternative_Asset":
                "Asset"
        }
    )
)

daily_export = make_excel_safe(
    daily_export
)

coverage_export = make_excel_safe(
    coverage
)

variation_export = make_excel_safe(
    variation_summary
)

market_info_export = make_excel_safe(
    market_info
)

token_map_export = make_excel_safe(
    token_map
)

feature_guide_export = make_excel_safe(
    feature_guide
)

not_historical_export = make_excel_safe(
    not_historical
)

qc_export = make_excel_safe(
    qc_summary
)

failed_windows_export = make_excel_safe(
    failed_windows
)


# ------------------------------------------------------------
# Save Excel workbook
# ------------------------------------------------------------

with pd.ExcelWriter(
    OUTPUT_XLSX,
    engine="openpyxl"
) as writer:

    master_plus_export.to_excel(
        writer,
        sheet_name="Model_Ready_Master",
        index=False
    )

    daily_export.to_excel(
        writer,
        sheet_name="Daily_PM_Activity",
        index=False
    )

    coverage_export.to_excel(
        writer,
        sheet_name="Coverage",
        index=False
    )

    variation_export.to_excel(
        writer,
        sheet_name="Within_Event_Variation",
        index=False
    )

    market_info_export.to_excel(
        writer,
        sheet_name="Market_Info",
        index=False
    )

    token_map_export.to_excel(
        writer,
        sheet_name="Token_Map",
        index=False
    )

    feature_guide_export.to_excel(
        writer,
        sheet_name="Feature_Guide",
        index=False
    )

    not_historical_export.to_excel(
        writer,
        sheet_name="Unavailable_Historical",
        index=False
    )

    qc_export.to_excel(
        writer,
        sheet_name="QC_Summary",
        index=False
    )

    failed_windows_export.to_excel(
        writer,
        sheet_name="Failed_API_Windows",
        index=False
    )


# ------------------------------------------------------------
# Save raw trades separately as compressed CSV
# CSV can retain the UTC timestamps
# ------------------------------------------------------------

raw_export_cols = [
    c for c in [
        "proxyWallet",
        "Side",
        "Token_ID",
        "Condition_ID",
        "Trade_Size_Shares",
        "Trade_Price",
        "Derived_Notional",
        "Timestamp",
        "Datetime_UTC",
        "Date",
        "outcome",
        "transactionHash"
    ]
    if c in trades.columns
]

trades[
    raw_export_cols
].to_csv(
    RAW_TRADES_OUTPUT,
    index=False,
    compression="gzip"
)


print("Saved successfully:")
print(OUTPUT_XLSX)

print("\nRaw historical trades:")
print(RAW_TRADES_OUTPUT)

## Step 25 — Download the Files


In [ ]:
# ============================================================
# STEP 25: Download Outputs
# ============================================================

from google.colab import files

files.download(
    "/content/"
    "Final_Competitive_Markets_MODEL_READY_"
    "with_PM_Trading_Activity.xlsx"
)

files.download(
    "/content/"
    "Polymarket_Selected_Markets_"
    "Historical_Trades.csv.gz"
)
